In [ ]:
from pathlib import Path
import json

from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley

pdb_dir  = Path("../../../data/rBMRB/cluster_30p/structures_split_filtered/test")
src_json = Path("../../../data/analysis/Fig4/dssp_results/per_residue_records.json")
out_dir  = Path("../../../data/analysis/Fig4/sasa_results")

# three-letter -> one-letter
AA3TO1 = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C',
    'GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I',
    'LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P',
    'SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
    'HID':'H','HIE':'H','HIP':'H','HSD':'H','HSE':'H',
    'MSE':'M','SEC':'C',
}
# backbone amide H atom names (differ between force fields; N-terminal H1/H2/H3 are not amide protons and are excluded)
BACKBONE_H_NAMES = {"H", "HN"}

NH_SASA_CUTOFF = 0.0   # nh_sasa <= cutoff -> buried (default of update_per_residue_sasa_category.py)


def biopython_nh_sasa(pdb_path, probe_radius=1.4, n_points=960):
    """Per-residue SASA (A^2) of the backbone N atom + amide H atom.
    Returns {(chain, resnum, inscode): {"n_sasa","h_sasa","nh_sasa","has_h"}}."""
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("sasa", pdb_path)

    sr = ShrakeRupley(probe_radius=probe_radius, n_points=n_points)
    sr.compute(struct, level="A")

    result = {}
    for model in struct:
        for chain in model:
            for res in chain:
                if res.id[0] != " ":            # skip HETATM / water
                    continue
                if res.resname not in AA3TO1:   # skip non-standard residues
                    continue

                n_sasa = res["N"].sasa if "N" in res else 0.0
                h_atom = next((res[name] for name in BACKBONE_H_NAMES if name in res), None)
                has_h  = h_atom is not None
                h_sasa = float(h_atom.sasa) if has_h else -1.0
                nh_sasa = float(n_sasa) + (h_sasa if has_h else 0.0)

                key = (chain.id.strip(), res.id[1], res.id[2].strip())
                result[key] = {"n_sasa": float(n_sasa), "h_sasa": h_sasa,
                               "nh_sasa": nh_sasa, "has_h": has_h}
    return result


def simple_category(aa, has_h, nh_sasa, cutoff=NH_SASA_CUTOFF):
    """SASA-only amide classification:
    proline has no amide proton; no_amide_H means the H is absent from the structure (e.g. N-terminus);
    all others are exposed if the N+H SASA is > cutoff, otherwise buried."""
    if str(aa).strip().upper() == "P":
        return "proline"
    if (not has_h) or nh_sasa is None:
        return "no_amide_H"
    return "exposed" if round(float(nh_sasa), 3) > cutoff else "buried"

In [ ]:
from collections import Counter

per_residue_records = json.load(open(src_json))

updated = missing_sasa = 0
cat_counts = Counter()

for pdb_file in sorted(pdb_dir.glob("*.pdb")):
    entry = pdb_file.stem
    if entry not in per_residue_records:
        continue

    nh_sasa_map = biopython_nh_sasa(pdb_file)   # {(chain, resnum, inscode): {...}}

    for r in per_residue_records[entry]:
        # all structures are single chain 'A' without insertion codes, so keys match the resnum in dssp_results
        sd = nh_sasa_map.get(("A", r["resnum"], ""))
        if sd is None:
            r["exposure"] = None
            missing_sasa += 1
            continue

        cat = simple_category(r["residue_type"], sd["has_h"], sd["nh_sasa"])
        r["exposure"] = cat
        cat_counts[cat] += 1
        updated += 1

out_dir.mkdir(parents=True, exist_ok=True)
out_json = out_dir / "per_residue_records.json"
with open(out_json, "w") as f:
    json.dump(per_residue_records, f)

print(f"{updated} residues labeled, {missing_sasa} residues with no SASA match (left exposure=None)")
print(f"exposure category counts: {dict(cat_counts)}")
print(f"Saved to {out_json}")